# Seeing the Clouds Coming: 15-Minute Solar Power Forecasting from Sky Images

APS360 course project, Daksh Khanna.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DakshK26/aps360-solar-nowcasting/blob/main/notebooks/solar_nowcasting.ipynb)

**Task.** Given the last 15 minutes of fisheye sky images and PV power output from a 30.1 kW-DC rooftop array at Stanford, predict the array's power output 15 minutes ahead.

**Data.** SKIPP'D benchmark (Nie et al., 2023; CC BY 4.0), loaded from Hugging Face (`solarbench/SKIPPD`).

**Status.** Data cleaning, clear-sky modelling, forecast-sample construction, the smart-persistence and ridge baselines, and the model definition are implemented. Image data loading for training and the training loop are next (progress report).

In [ ]:
!pip -q install pvlib datasets

In [ ]:
import io

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pvlib
import torch
import torch.nn as nn
from datasets import load_dataset
from PIL import Image
from sklearn.linear_model import Ridge

SEED = 360
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

## 1. Load and clean the PV log

The `labels` config holds only `(time, pv)` pairs (a few MB), which is enough for the clear-sky model and the baselines.

**Timestamp correction.** The published timestamps are offset by one hour during daylight-saving months: the measured midday peak lags the computed solar peak by about 60 minutes in summer in all three years, but not in winter. The original logger clock followed local time (with DST) while the export treated it as fixed UTC-8. We recover the wall-clock reading and re-localize it as `US/Pacific`, which aligns measured and clear-sky peaks to within a few minutes year-round.

In [ ]:
labels = load_dataset("solarbench/SKIPPD", "labels")


def fix_time(col):
    wall = pd.to_datetime(col).dt.tz_convert("Etc/GMT+8").dt.tz_localize(None)
    local = wall.dt.tz_localize("US/Pacific", ambiguous="NaT", nonexistent="NaT")
    return local.dt.tz_convert("UTC").dt.round("min")


def clean_pv(df):
    out = pd.DataFrame({"time": fix_time(df["time"]), "pv": df["pv"].astype(float)})
    out = out[out["time"].notna() & (out["pv"] >= 0)]
    return out.groupby("time")["pv"].mean().sort_index()


raw = {s: labels[s].to_pandas() for s in ["train", "test"]}
pv_train, pv_test = clean_pv(raw["train"]), clean_pv(raw["test"])
print("pairs:", len(raw["train"]), len(raw["test"]), "-> unique minutes:", len(pv_train), len(pv_test))
print("days: train", pv_train.index.tz_convert("US/Pacific").normalize().nunique(),
      "test", pv_test.index.tz_convert("US/Pacific").normalize().nunique())

## 2. Clear-sky PV model

Ineichen clear-sky irradiance (pvlib) transposed onto the array plane (tilt 22.5 deg, azimuth 195 deg) and converted to DC power with PVWatts. A single scale factor calibrates the model to the array using the 98th percentile of `P / P_cs` on training data with solar zenith below 70 deg.

In [ ]:
SITE = pvlib.location.Location(latitude=37.4275, longitude=-122.1745, tz="UTC", altitude=30)
TILT, AZIMUTH, P_DC0 = 22.5, 195.0, 30.1


def clear_sky(times):
    sp = SITE.get_solarposition(times)
    cs = SITE.get_clearsky(times, model="ineichen", solar_position=sp)
    poa = pvlib.irradiance.get_total_irradiance(
        TILT, AZIMUTH, sp["apparent_zenith"], sp["azimuth"], cs["dni"], cs["ghi"], cs["dhi"])
    p = pvlib.pvsystem.pvwatts_dc(poa["poa_global"].fillna(0), temp_cell=25.0, pdc0=P_DC0, gamma_pdc=-0.004)
    return pd.DataFrame({"p_clear": p.clip(lower=0), "zenith": sp["apparent_zenith"],
                         "azimuth": sp["azimuth"]}, index=times)


cs_train, cs_test = clear_sky(pv_train.index), clear_sky(pv_test.index)
m = cs_train["zenith"] < 70
scale = np.nanquantile(pv_train[m] / cs_train.loc[m, "p_clear"], 0.98)
cs_train["p_clear"] *= scale
cs_test["p_clear"] *= scale
print("clear-sky calibration factor:", round(float(scale), 3))

## 3. Forecast samples

Anchor times `t` every 2 minutes. Each sample needs PV at `t-14, t-12, ..., t` (8 steps) and at `t+15`, with no gaps, and solar zenith below 85 deg at both `t` and `t+15`.

In [ ]:
HIST = np.arange(-14, 1, 2)
HORIZON = 15
MAX_ZENITH = 85.0


def build_samples(pv, cs, stride=2):
    idx = pv.index
    anchors = idx[(idx.minute % stride) == 0]
    offsets = [pd.Timedelta(minutes=int(h)) for h in HIST] + [pd.Timedelta(minutes=HORIZON)]
    ok = np.ones(len(anchors), dtype=bool)
    for off in offsets:
        ok &= (anchors + off).isin(idx)
    anchors = anchors[ok]
    target = anchors + pd.Timedelta(minutes=HORIZON)
    ok = (cs.loc[anchors, "zenith"].to_numpy() < MAX_ZENITH) & (cs.loc[target, "zenith"].to_numpy() < MAX_ZENITH)
    anchors, target = anchors[ok], target[ok]
    hist_t = [anchors + pd.Timedelta(minutes=int(h)) for h in HIST]
    return {
        "anchor": anchors,
        "hist_pv": np.stack([pv.loc[t].to_numpy() for t in hist_t], 1),
        "hist_cs": np.stack([cs.loc[t, "p_clear"].to_numpy() for t in hist_t], 1),
        "cs_target": cs.loc[target, "p_clear"].to_numpy(),
        "zen_target": cs.loc[target, "zenith"].to_numpy(),
        "azi_target": cs.loc[target, "azimuth"].to_numpy(),
        "y": pv.loc[target].to_numpy(),
    }


tr, te = build_samples(pv_train, cs_train), build_samples(pv_test, cs_test)
print("forecast samples: train", len(tr["y"]), "test", len(te["y"]))

## 4. Baseline: smart persistence

The clear-sky index `k = P / P_cs` is assumed constant over the horizon: `P_hat(t+15) = k(t) * P_cs(t+15)`. Test days are split into sunny and cloudy halves by their mean absolute minute-to-minute change in `k`.

In [ ]:
EPS = 0.5


def clear_sky_index(p, p_cs):
    return np.clip(np.where(p_cs > EPS, p / np.maximum(p_cs, EPS), 1.0), 0.0, 1.5)


def smart_persistence(s):
    return clear_sky_index(s["hist_pv"][:, -1], s["hist_cs"][:, -1]) * s["cs_target"]


def rmse(y, p):
    return float(np.sqrt(np.mean((y - p) ** 2)))


def mae(y, p):
    return float(np.mean(np.abs(y - p)))


def forecast_skill(y, p, p_ref):
    return 1.0 - rmse(y, p) / rmse(y, p_ref)


k_test = pv_test / cs_test["p_clear"].clip(lower=EPS)
variability = k_test.diff().abs().groupby(k_test.index.tz_convert("US/Pacific").normalize()).mean()
cloudy_days = set(variability[variability > variability.median()].index)
is_cloudy = np.array([d in cloudy_days for d in te["anchor"].tz_convert("US/Pacific").normalize()])

p_sp = smart_persistence(te)


def report(name, p):
    print(f"{name:>18s}  RMSE {rmse(te['y'], p):.3f} kW  MAE {mae(te['y'], p):.3f} kW  FS {forecast_skill(te['y'], p, p_sp):+.3f}")
    for cond, msk in [("sunny", ~is_cloudy), ("cloudy", is_cloudy)]:
        print(f"{cond:>18s}  RMSE {rmse(te['y'][msk], p[msk]):.3f} kW  FS {forecast_skill(te['y'][msk], p[msk], p_sp[msk]):+.3f}")


report("smart persistence", p_sp)

## 5. Baseline: ridge regression on PV history (no images)

Predicts `k(t+15)` from the 8-step clear-sky-index history and sun geometry at `t+15`. Comparing the deep model against this isolates the value of the sky images.

In [ ]:
def tabular_features(s):
    k_hist = clear_sky_index(s["hist_pv"], s["hist_cs"])
    geo = np.stack([np.cos(np.radians(s["zen_target"])),
                    np.sin(np.radians(s["azi_target"])),
                    np.cos(np.radians(s["azi_target"])),
                    s["cs_target"] / P_DC0], 1)
    return np.concatenate([k_hist, geo], 1)


ridge = Ridge(alpha=1.0).fit(tabular_features(tr), clear_sky_index(tr["y"], tr["cs_target"]))
p_ridge = np.clip(ridge.predict(tabular_features(te)), 0.0, 1.5) * te["cs_target"]
report("ridge (PV only)", p_ridge)

## 6. Sky images

The `default` config adds the 64x64 RGB sky image to each `(time, pv)` pair (about 2.2 GB for train, 90 MB for test).

In [ ]:
img_test = load_dataset("solarbench/SKIPPD", "default", split="test")
times = fix_time(pd.Series(img_test["time"]))
pos = pd.Series(np.arange(len(times)), index=times)
pos = pos[pos.index.notna() & ~pos.index.duplicated()]

start = pd.Timestamp("2018-01-25 12:59", tz="US/Pacific").tz_convert("UTC")
fig, axes = plt.subplots(1, 4, figsize=(10, 2.8))
for ax, m in zip(axes, [0, 4, 8, 12]):
    ts = start + pd.Timedelta(minutes=m)
    row = img_test[int(pos[ts])]
    ax.imshow(row["image"])
    ax.set_title(f"{ts.tz_convert('US/Pacific'):%H:%M}  P = {row['pv']:.1f} kW")
    ax.axis("off")
plt.tight_layout()

## 7. Model: CNN frame encoder + LSTM + PV fusion

A shared 4-block CNN embeds each of the 8 frames; an LSTM summarizes the sequence; its final state is concatenated with an MLP embedding of the clear-sky-index history, clear-sky PV history, `P_cs(t+15)` and sun geometry; an MLP head regresses normalized power `P(t+15) / 30.1 kW`. Predicting `k(t+15)` instead is planned as an ablation (SUNSET found direct PV targets worked better).

In [ ]:
class FrameEncoder(nn.Module):
    def __init__(self, emb=128):
        super().__init__()
        layers, c = [], 3
        for out in [32, 64, 128, 128]:
            layers += [nn.Conv2d(c, out, 3, padding=1), nn.BatchNorm2d(out), nn.ReLU(inplace=True), nn.MaxPool2d(2)]
            c = out
        self.conv = nn.Sequential(*layers)
        self.proj = nn.Linear(128 * 4 * 4, emb)

    def forward(self, x):
        return self.proj(self.conv(x).flatten(1))


class SkyNowcaster(nn.Module):
    def __init__(self, n_hist=8, n_aux=4, emb=128, hidden=128):
        super().__init__()
        self.encoder = FrameEncoder(emb)
        self.lstm = nn.LSTM(emb, hidden, batch_first=True)
        self.aux = nn.Sequential(nn.Linear(2 * n_hist + n_aux, 64), nn.ReLU())
        self.head = nn.Sequential(nn.Dropout(0.3), nn.Linear(hidden + 64, 64), nn.ReLU(), nn.Linear(64, 1))

    def forward(self, frames, aux):
        b, t = frames.shape[:2]
        z = self.encoder(frames.flatten(0, 1)).view(b, t, -1)
        _, (h, _) = self.lstm(z)
        return self.head(torch.cat([h[-1], self.aux(aux)], 1)).squeeze(1)


net = SkyNowcaster().to(device)
out = net(torch.randn(4, 8, 3, 64, 64, device=device), torch.randn(4, 20, device=device))
print("output shape:", tuple(out.shape), "| parameters:", sum(p.numel() for p in net.parameters()))

## 8. Next steps (progress report)

- Build a PyTorch `Dataset` that gathers the 8 frames for each forecast anchor from the `default` config.
- Hold out ~10% of training days (stratified sunny/cloudy) for validation; oversample cloudy samples.
- Train with Huber loss on normalized `P(t+15)`, Adam, weight decay, photometric augmentation (brightness/contrast only), early stopping on validation forecast skill.
- Compare against both baselines on the 20 official test days; ablations: ConvLSTM temporal module, clear-sky-index target, multi-horizon outputs.